In [1]:
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

In [3]:
filenames = {
    'all'       : './catalogs/0315_all_test.fits',
    'dps'       : './catalogs/0315_dps_test.fits',
    'cs'        : './catalogs/0315_cs_test.fits',
    'nbcs'      : './catalogs/0315_nbcs_test.fits',
    'cs-nbcs'   : './catalogs/0315_cs-nbcs_test.fits'
}

In [4]:
line_cols = ['OII3726', 'OII3729',
        'Hbeta',
        'OIII4959', 'OIII5007',
        'NII6548', 'Halpha', 'NII6583', 
        'SII6716', 'SII6731']
dp_cols = [f'{col}_DP' for col in line_cols]
dp_snr_cols = [f'{col}_SNR' for col in line_cols]
flux_1comp_cols = [f'{col}_1COMP' for col in line_cols]
flux_2compL_cols = [f'{col}_FLUX_2COMP_L' for col in line_cols]
flux_2compR_cols = [f'{col}_FLUX_2COMP_R' for col in line_cols]

# Select parent sources who have at least one significant emission line (>5 S/N ratio)

In [5]:
FIT = FitSpectrum()
DP = DP()

ALL_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=None, subtype_filter='QSO')
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Data loading and processing took 43.02 seconds.
Total number of spectra: 99812


In [6]:
# ALL_SPECTRA = FIT.label_emission_lines(ALL_SPECTRA, 5)
# ALL_SPECTRA = FIT.significant_emission_filter(ALL_SPECTRA)
# print('Total number of spectra:', len(ALL_SPECTRA.targetID))

In [7]:
# ALL_SPECTRA = ALL_SPECTRA.shrink_dataset(10)
# print('Number of spectra after shrinking:', len(ALL_SPECTRA.targetID))

In [8]:
# ALL_SPECTRA.df.head()

In [9]:
# dp_parent, model_1comp_all, left_2comp_all, right_2comp_all = DP.fit_all(data_class=ALL_SPECTRA, n_jobs=10)

In [10]:
# DP.get_catalog(df=dp_parent, fname=filenames['all'], model_1comp=model_1comp_all, left_2comp=left_2comp_all, right_2comp=right_2comp_all)

In [11]:
dp_parent, model_1comp_all, left_2comp_all, right_2comp_all = DP.extract_fits_data(filenames['all'])
dp_parent = DP.bpt_classification(dp_parent, two_comp=True)
dp_parent = DP.estimate_SFR(dp_parent, two_comp=True)
DP.get_catalog(df=dp_parent, fname=filenames['all'], model_1comp=model_1comp_all, left_2comp=left_2comp_all, right_2comp=right_2comp_all)


/Users/hyp0515/master_project/code/main/nature_of_DP/src/DP.py:436: RuntimeWarning: divide by zero encountered in log10
  ebv = 1.97 * np.log10((flux_halpha/flux_hbeta) / 2.86)
/Users/hyp0515/master_project/code/main/nature_of_DP/src/DP.py:438: RuntimeWarning: overflow encountered in scalar power
  flux_intrinsic = flux_halpha / (10**(A_halpha/-2.5)) * 1e-17
/Users/hyp0515/master_project/code/main/nature_of_DP/src/DP.py:472: RuntimeWarning: divide by zero encountered in log10
  sfr_1comp = np.where((sfr_1comp > 0), np.log10(sfr_1comp), -15)
/Users/hyp0515/master_project/code/main/nature_of_DP/src/DP.py:478: RuntimeWarning: divide by zero encountered in log10
  sfr_2comp = np.where((sfr_2comp > 0), np.log10(sfr_2comp), -15)


In [12]:
dp_sample, model_1comp_dps, left_2comp_dps, right_2comp_dps = DP.select_dp_sample(dp_parent, model_1comp_all, left_2comp_all, right_2comp_all)
DP.get_catalog(df=dp_sample, fname=filenames['dps'], model_1comp=model_1comp_dps, left_2comp=left_2comp_dps, right_2comp=right_2comp_dps)

In [13]:
dp_parent, model_1comp, left_2comp, right_2comp = DP.extract_fits_data(filenames['all'])
dps_df, _, _, _ = DP.extract_fits_data(filenames['dps'])
cs_df, nbcs_df, cs_nbcs_df = DP.select_nbcs(dp_parent=dp_parent, dp_sample=dps_df)

cs_df.drop(columns=dp_cols+flux_2compL_cols+flux_2compR_cols+['LOGSFR_2COMP'], inplace=True)
nbcs_df.drop(columns=dp_cols+flux_2compL_cols+flux_2compR_cols+['LOGSFR_2COMP'], inplace=True)
cs_nbcs_df.drop(columns=dp_cols+flux_2compL_cols+flux_2compR_cols+['LOGSFR_2COMP'], inplace=True)

DP.get_catalog(cs_df, model_1comp=model_1comp[cs_df.index], left_2comp=left_2comp[cs_df.index], right_2comp=right_2comp[cs_df.index], fname=filenames['cs'])
DP.get_catalog(nbcs_df, model_1comp=model_1comp[nbcs_df.index], left_2comp=left_2comp[nbcs_df.index], right_2comp=right_2comp[nbcs_df.index], fname=filenames['nbcs'])
DP.get_catalog(cs_nbcs_df, model_1comp=model_1comp[cs_nbcs_df.index], left_2comp=left_2comp[cs_nbcs_df.index], right_2comp=right_2comp[cs_nbcs_df.index], fname=filenames['cs-nbcs'])

In [14]:
cs_df.columns

Index(['TARGETID', 'RA', 'DEC', 'Z', 'LOGM', 'LOGSFR', 'DV_R', 'DV_L',
       'SIGMA_R', 'SIGMA_L', 'SIGMA_1COMP', 'P_VALUE', 'OII3726_FLUX_1COMP',
       'OII3729_FLUX_1COMP', 'Hbeta_FLUX_1COMP', 'OIII4959_FLUX_1COMP',
       'OIII5007_FLUX_1COMP', 'NII6548_FLUX_1COMP', 'Halpha_FLUX_1COMP',
       'NII6583_FLUX_1COMP', 'SII6716_FLUX_1COMP', 'SII6731_FLUX_1COMP',
       'OII3726_3NOISE', 'OII3729_3NOISE', 'Hbeta_3NOISE', 'OIII4959_3NOISE',
       'OIII5007_3NOISE', 'NII6548_3NOISE', 'Halpha_3NOISE', 'NII6583_3NOISE',
       'SII6716_3NOISE', 'SII6731_3NOISE', 'OII3726_SNR', 'OII3729_SNR',
       'Hbeta_SNR', 'OIII4959_SNR', 'OIII5007_SNR', 'NII6548_SNR',
       'Halpha_SNR', 'NII6583_SNR', 'SII6716_SNR', 'SII6731_SNR', 'BPT_1COMP',
       'BPT_2COMP_L', 'BPT_2COMP_R', 'BPT_2COMP', 'LOGSFR_1COMP'],
      dtype='object')

In [15]:
dps_df.columns

Index(['TARGETID', 'RA', 'DEC', 'Z', 'LOGM', 'LOGSFR', 'DV_R', 'DV_L',
       'SIGMA_R', 'SIGMA_L', 'SIGMA_1COMP', 'P_VALUE', 'OII3726_FLUX_1COMP',
       'OII3729_FLUX_1COMP', 'Hbeta_FLUX_1COMP', 'OIII4959_FLUX_1COMP',
       'OIII5007_FLUX_1COMP', 'NII6548_FLUX_1COMP', 'Halpha_FLUX_1COMP',
       'NII6583_FLUX_1COMP', 'SII6716_FLUX_1COMP', 'SII6731_FLUX_1COMP',
       'OII3726_FLUX_2COMP_L', 'OII3729_FLUX_2COMP_L', 'Hbeta_FLUX_2COMP_L',
       'OIII4959_FLUX_2COMP_L', 'OIII5007_FLUX_2COMP_L',
       'NII6548_FLUX_2COMP_L', 'Halpha_FLUX_2COMP_L', 'NII6583_FLUX_2COMP_L',
       'SII6716_FLUX_2COMP_L', 'SII6731_FLUX_2COMP_L', 'OII3726_FLUX_2COMP_R',
       'OII3729_FLUX_2COMP_R', 'Hbeta_FLUX_2COMP_R', 'OIII4959_FLUX_2COMP_R',
       'OIII5007_FLUX_2COMP_R', 'NII6548_FLUX_2COMP_R', 'Halpha_FLUX_2COMP_R',
       'NII6583_FLUX_2COMP_R', 'SII6716_FLUX_2COMP_R', 'SII6731_FLUX_2COMP_R',
       'OII3726_DP', 'OII3729_DP', 'Hbeta_DP', 'OIII4959_DP', 'OIII5007_DP',
       'NII6548_DP', 'Halpha_D